In [ ]:
import os
import sys
import subprocess
import json

from google.colab import drive, userdata
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

REPO_DIR = "/content/teachmewrite"
REPO_URL = "https://github.com/edmedpublic-hub/teachmewrite.git"

if os.path.exists(REPO_DIR):
    print("Repo exists. Pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], capture_output=True)
else:
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "anthropic"], check=True)

try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
    print("Secrets loaded successfully.")
except Exception as e:
    print(f"ERROR loading secrets: {e}")
    raise

LIVE_URL = "https://teachmewrite-ai-195673983724.asia-south1.run.app"

assert os.environ.get("ANTHROPIC_API_KEY") is not None, "ANTHROPIC_API_KEY is missing!"
print(f"Target Live Service: {LIVE_URL}")
print("✅ Cell 1a complete. Environment is ready.")

Mounted at /content/drive
Cloning repo...
Installing dependencies...
Secrets loaded successfully.
Target Live Service: https://teachmewrite-ai-195673983724.asia-south1.run.app
✅ Cell 1a complete. Environment is ready.


In [ ]:
import requests
from anthropic import Anthropic

# Initialize the Claude client
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# 1. Define the Tool Schema (How the LLM sees the tool)
tools = [
    {
        "name": "get_writing_feedback",
        "description": "Get automated English writing feedback for a student's text. Use this before asking the student a question.",
        "input_schema": {
            "type": "object",
            "properties": {
                "student_text": {
                    "type": "string",
                    "description": "The English text written by the student."
                },
                "level": {
                    "type": "integer",
                    "description": "The student's English proficiency level (1 to 7)."
                }
            },
            "required": ["student_text", "level"]
        }
    }
]

# 2. Define the actual Python function that executes when the LLM calls the tool
def get_writing_feedback(student_text: str, level: int) -> dict:
    """Calls the live Cloud Run endpoint to get feedback."""
    print(f"  [Tool Executing] Calling /feedback for level {level}...")
    payload = {"student_text": student_text, "level": level}
    try:
        resp = requests.post(f"{LIVE_URL}/feedback", json=payload, timeout=45)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        return {"error": str(e)}

print("✅ Cell 1b complete. Tool schema and execution function are ready.")

✅ Cell 1b complete. Tool schema and execution function are ready.


In [ ]:
def socratic_tutor_agent(student_text: str, level: int, max_iterations: int = 3):
    """
    An agent that calls the writing feedback tool, analyzes the issues,
    and asks the student a Socratic follow-up question.
    """
    system_prompt = """You are a Socratic English tutor for Pakistani students.
Your goal is to help the student discover their own errors, not just give them the answers.

Instructions:
1. ALWAYS call the `get_writing_feedback` tool first to understand the student's errors.
2. After receiving the feedback, DO NOT simply tell the student their errors.
3. Instead, pick ONE major issue (preferably grammar or clarity) and ask a guiding question.
4. Example: If the feedback says "I am agree should be I agree", ask: "I noticed the phrase 'I am agree'. Do you know what the correct verb form is when expressing agreement?"
5. Keep the tone encouraging, professional, and brief.
"""

    messages = [
        {"role": "user", "content": f"Here is my writing: '{student_text}' (Level {level}). Help me improve it."}
    ]

    for iteration in range(max_iterations):
        print(f"  [Agent Iteration {iteration + 1}] Calling Claude...")

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            system=system_prompt,
            tools=tools,
            messages=messages
        )

        # Check if Claude wants to use a tool
        if response.stop_reason == "tool_use":
            # Extract the tool use block
            tool_use_block = next(b for b in response.content if b.type == "tool_use")

            tool_name = tool_use_block.name
            tool_input = tool_use_block.input
            tool_use_id = tool_use_block.id

            print(f"  [Agent] Decided to call tool: {tool_name}")

            # Execute the tool
            if tool_name == "get_writing_feedback":
                tool_result = get_writing_feedback(**tool_input)
            else:
                tool_result = {"error": f"Unknown tool: {tool_name}"}

            # Append Claude's tool-use response AND our tool result to messages
            messages.append({"role": "assistant", "content": response.content})
            messages.append({
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": tool_use_id,
                        "content": json.dumps(tool_result)
                    }
                ]
            })

        elif response.stop_reason == "end_turn":
            # Claude finished — extract the text response
            final_text = "".join(b.text for b in response.content if hasattr(b, "text"))
            return final_text

        else:
            return f"Agent stopped unexpectedly. Reason: {response.stop_reason}"

    return "Agent reached max iterations without finishing."

print("✅ Cell 1c complete. socratic_tutor_agent() is ready.")

✅ Cell 1c complete. socratic_tutor_agent() is ready.


In [ ]:
student_text = "I am agree with my father that education is more better than money."
student_level = 5

print(f"--- Running Socratic Agent ---")
print(f"Student text: {student_text}")
print(f"Level: {student_level}\n")

# Run the agent
agent_response = socratic_tutor_agent(student_text, student_level)

print(f"\n--- Final Agent Response ---")
print(agent_response)

assert len(agent_response) > 20, "Agent response too short!"
assert "I am agree" in agent_response or "agree" in agent_response.lower(), "Agent should mention the grammar error"
print("\n✅ Cell 1d complete. Agent ran successfully.")

--- Running Socratic Agent ---
Student text: I am agree with my father that education is more better than money.
Level: 5

  [Agent Iteration 1] Calling Claude...
  [Agent] Decided to call tool: get_writing_feedback
  [Tool Executing] Calling /feedback for level 5...
  [Agent Iteration 2] Calling Claude...

--- Final Agent Response ---
Great effort! I can see you're expressing a meaningful idea. I'd like to help you improve two grammar issues.

Let me start with the first one:

**Look at this phrase: "I am agree"**

Think about this: when you say "I am," you're adding an auxiliary verb. Can you think of other action verbs that use "am"? For example, do we say "I am *understand*" or "I am *run*"? 

What do you think is the correct way to say it when you express agreement? 🤔

✅ Cell 1d complete. Agent ran successfully.


In [ ]:
agent_artifact = {
    "student_text": student_text,
    "level": student_level,
    "agent_response": agent_response,
    "model": "claude-haiku-4-5-20251001",
    "tool_used": "get_writing_feedback",
    "tool_endpoint": f"{LIVE_URL}/feedback"
}

save_path = "/content/drive/MyDrive/teachmewrite-ai/agent_day91.json"
with open(save_path, "w") as f:
    json.dump(agent_artifact, f, indent=2)

print(f"Saved agent artifact to {save_path}")
assert os.path.exists(save_path), "Save failed!"
print("✅ Day 91 artifact saved.")

Saved agent artifact to /content/drive/MyDrive/teachmewrite-ai/agent_day91.json
✅ Day 91 artifact saved.
